In [1]:
import sys, math, time
import numpy as np
import pygame

# Make sure CARLA PythonAPI is on the path
sys.path.append(r"C:\Users\natha\Downloads\CARLA_0.9.16\PythonAPI")
sys.path.append(r"C:\Users\natha\Downloads\CARLA_0.9.16\PythonAPI\carla\dist")

import carla

c:\Users\natha\Downloads\CARLA repo\HondaAI\CARLA-sim\.carla-venv\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.12.10)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
# Core elements in carla are: actors, blueprints, worlds, and clients
client = carla.Client('localhost', 2000)

# Set the map to Town06. Silence output
try:
    client.load_world('Town06')
except RuntimeError:
    pass  # Ignore timeout or connection errors

In [3]:
world = client.get_world()
blueprints = world.get_blueprint_library()
spectator = world.get_spectator()

print("World loaded:", world.get_map().name)

World loaded: Carla/Maps/Town06


In [4]:
# Clean up leftover vehicles/sensors
for actor in world.get_actors():
    if actor.type_id.startswith("vehicle") or actor.type_id.startswith("sensor"):
        actor.destroy()

print("Destroyed previous vehicles/sensors.")

# Spectator top-down
top_loc = carla.Location(x=116, y=244.485397, z=100.0)
top_rot = carla.Rotation(pitch=270.0, yaw=270.0, roll=0.0)
spectator.set_transform(carla.Transform(top_loc, top_rot))

# Ego (hero) blueprint + spawn
ego_bp = blueprints.find('vehicle.dodge.charger_2020')
ego_bp.set_attribute('role_name', 'hero')

ego_spawn_loc = carla.Location(x=21, y=244.485397, z=0.5)
ego_spawn_rot = carla.Rotation(pitch=0.0, yaw=0.0, roll=0.0)
ego_tf = carla.Transform(ego_spawn_loc, ego_spawn_rot)

# NPC (police) blueprint + spawn (left lane: y - 3.5)
npc_bp = blueprints.find('vehicle.dodge.charger_police_2020')
npc_spawn_loc = carla.Location(x=15, y=244.485397 - 3.5, z=0.5)  # slightly behind hero
npc_spawn_rot = carla.Rotation(pitch=0.0, yaw=0.0, roll=0.0)
npc_tf = carla.Transform(npc_spawn_loc, npc_spawn_rot)

ego = world.try_spawn_actor(ego_bp, ego_tf)
npc = world.try_spawn_actor(npc_bp, npc_tf)

print("Ego:", ego)
print("NPC:", npc)

# Traffic Manager for NPC autopilot
tm = client.get_trafficmanager()
tm_port = tm.get_port()

npc.set_autopilot(True, tm_port)  # NPC uses TM (keeps lane/path)
ego.set_autopilot(False)          # Ego stays manual

print("Traffic Manager port:", tm_port)


Destroyed previous vehicles/sensors.
Ego: Actor(id=120, type=vehicle.dodge.charger_2020)
NPC: Actor(id=121, type=vehicle.dodge.charger_police_2020)
Traffic Manager port: 8000


In [5]:
# Attach a driver's POV camera to the hero
W, H = 800, 450

cam_bp = blueprints.find("sensor.camera.rgb")
cam_bp.set_attribute("image_size_x", str(W))
cam_bp.set_attribute("image_size_y", str(H))
cam_bp.set_attribute("fov", "90")

cam_tf = carla.Transform(
    carla.Location(x=0.2, y=-0.36, z=1.2),
    carla.Rotation(pitch=-10.0, yaw=0.0, roll=0.0)
)

camera = world.spawn_actor(cam_bp, cam_tf, attach_to=ego)

# ----- Pygame setup -----
pygame.init()
print("pygame.get_init():", pygame.get_init())

screen = pygame.display.set_mode((W, H))
pygame.display.set_caption("Hero POV")
clock = pygame.time.Clock()

print("display surface:", pygame.display.get_surface())

latest_image = None

def on_image(img):
    global latest_image
    array = np.frombuffer(img.raw_data, dtype=np.uint8)
    array = array.reshape((img.height, img.width, 4))
    latest_image = array[:, :, :3]  # RGB

camera.listen(on_image)

print("Camera attached and Pygame window should now be visible.")


pygame.get_init(): True
display surface: <Surface(800x450x32 SW)>
Camera attached and Pygame window should now be visible.


In [ ]:
def get_speed(vehicle: carla.Vehicle) -> float:
    v = vehicle.get_velocity()
    return math.sqrt(v.x**2 + v.y**2 + v.z**2)  # m/s

running = True
print("WASD to drive, ESC to quit this loop.")

while running:
    # Safety: if pygame/display is gone, exit cleanly
    if not pygame.get_init() or not pygame.display.get_surface():
        print("Pygame window closed — stopping loop.")
        break

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            print("QUIT event received.")
            running = False

    keys = pygame.key.get_pressed()

    throttle = 0.0
    brake = 0.0
    steer = 0.0

    if keys[pygame.K_w]:
        throttle = 0.6
    if keys[pygame.K_s]:
        brake = 1.0
    if keys[pygame.K_a]:
        steer = -0.4
    if keys[pygame.K_d]:
        steer = 0.4
    if keys[pygame.K_ESCAPE]:
        print("ESC pressed, exiting loop.")
        running = False

    # Apply control to hero
    ego.apply_control(carla.VehicleControl(
        throttle=throttle,
        brake=brake,
        steer=steer
    ))

    # NPC: keep autopilot, just match desired speed with ego
    hero_speed = get_speed(ego)       # m/s
    hero_speed_kmh = hero_speed * 3.6 # Traffic Manager uses km/h

    tm.vehicle_desired_speed(npc, hero_speed_kmh)

    # Draw latest camera image if available
    if latest_image is not None:
        surf = pygame.surfarray.make_surface(np.rot90(latest_image))
        screen.blit(surf, (0, 0))

    pygame.display.flip()
    clock.tick(30)  # ~30 FPS

print("Control loop exited.")


WASD to drive, ESC to quit this loop.


AttributeError: 'TrafficManager' object has no attribute 'vehicle_desired_speed'

: 

In [12]:
try:
    camera.stop()
except Exception:
    pass

for actor in world.get_actors():
    if actor.type_id.startswith("vehicle") or actor.type_id.startswith("sensor"):
        actor.destroy()

pygame.quit()
print("Cleaned up vehicles, sensors, and pygame.")


Cleaned up vehicles, sensors, and pygame.
